# EyePACS: exploratory data analysis

Thin notebook over `dr.eda` and `dr.data`. Run `python -m dr.data` first, which writes
`data/processed/manifest.csv` and `data_quality.json`.

**Do not commit this notebook with outputs.** The example cells show licensed photographs.

In [ ]:
import json, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from dr import eda
from dr.config import load_config, resolve_path

cfg = load_config()
RAW = resolve_path(cfg, "paths.raw_dir")
PROCESSED = resolve_path(cfg, "paths.processed_dir")
df = eda.load_manifest(PROCESSED / "manifest.csv")
report = json.loads((PROCESSED / "data_quality.json").read_text())
print(f"{len(df):,} images, {df.patient_id.nunique():,} patients, referable {df.label.mean():.2%}")
print("validation issues:", {k: v for k, v in report["issues"].items() if k.endswith(("count", "rows")) or k == "images_without_label"})

## Grade distribution

In [ ]:
eda.plot_grades(df);

## Image quality flags
Thresholds are in `configs/base.yaml` → `quality`.

In [ ]:
print(json.dumps(report["quality_counts"], indent=1), "\npoor share:", report["poor_quality_share"])
eda.plot_quality(df);

## Image statistics
These are also the image features monitored for drift after deployment.

In [ ]:
eda.plot_stat_histograms(df);

## Resolution and camera groups
The resized copy fixes the width at 1024 px, but the aspect ratio still identifies the camera type.

In [ ]:
eda.plot_resolutions(df)
eda.plot_referable_by_group(df);

## Example images per grade *(display only)*

In [ ]:
eda.show_examples(df[df.quality_ok], RAW, by='level', n=5, seed=1);

## Examples of poor-quality images *(display only)*

In [ ]:
eda.show_examples(df[~df.quality_ok], RAW, by='quality', n=5, seed=1);

## Duplicates
Files that share an MD5 are grouped together in the patient-level split (Phase 3).

In [ ]:
report['md5_duplicates']